In [4]:
from tensorflow.keras.models import load_model, Model

cnn_model = load_model("cnn_best_model.keras")
densenet_model = load_model("densenet121_best.keras")

print("Models loaded successfully!")

Models loaded successfully!


In [6]:
import tensorflow as tf

dummy = tf.random.normal((1, 224, 224, 3))

_ = cnn_model(dummy)
_ = densenet_model(dummy)

print("Both models are built.")

Both models are built.


In [7]:
from tensorflow.keras.models import Model

cnn_feature_extractor = Model(
    inputs=cnn_model.inputs,
    outputs=cnn_model.layers[-2].output
)

densenet_feature_extractor = Model(
    inputs=densenet_model.inputs,
    outputs=densenet_model.layers[-2].output
)

cnn_feature_extractor.trainable = False
densenet_feature_extractor.trainable = False

print("CNN:", cnn_feature_extractor.output_shape)
print("DenseNet:", densenet_feature_extractor.output_shape)

CNN: (None, 128)
DenseNet: (None, 256)


In [8]:
cnn_feature_extractor.trainable = False
densenet_feature_extractor.trainable = False

In [9]:
from tensorflow.keras.layers import (
    Input,
    Concatenate,
    BatchNormalization,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model

input_layer = Input(shape=(224, 224, 3))

# Feature Extraction
cnn_features = cnn_feature_extractor(input_layer)
densenet_features = densenet_feature_extractor(input_layer)

# Merge Features
merged = Concatenate()([cnn_features, densenet_features])

# Classification Head
x = BatchNormalization()(merged)

x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(7, activation="softmax")(x)

hybrid_model = Model(inputs=input_layer, outputs=output)

hybrid_model.summary()

Model: "functional_40"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_38       │ (None, 128)       │ 13,268,544 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_39       │ (None, 256)       │  7,299,904 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ functional_38[0]… │
│ (Concatenate)       │                   │            │ functional_39[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 384)       │      1,536 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     98,560 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 7)         │        903 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,702,343 (78.97 MB)

 Trainable params: 133,127 (520.03 KB)

 Non-trainable params: 20,569,216 (78.47 MB)

In [10]:
from tensorflow.keras.optimizers import Adam

hybrid_model.compile(
    optimizer=Adam(learning_rate=0.0002),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Hybrid CNN + DenseNet121 Compiled Successfully!")

Hybrid CNN + DenseNet121 Compiled Successfully!


In [12]:
print("train_ds" in globals())
print("val_ds" in globals())
print("test_ds" in globals())

False
False
False


In [13]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1811 files belonging to 7 classes.
Found 362 files belonging to 7 classes.
Found 365 files belonging to 7 classes.


In [15]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        verbose=1
    ),

    ModelCheckpoint(
        "cnn_densenet121_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks Ready!")

Callbacks Ready!


In [16]:
history = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20


57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1875 - loss: 2.5460
Epoch 1: val_accuracy improved from None to 0.71547, saving model to cnn_densenet121_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 167s 3s/step - accuracy: 0.2385 - loss: 2.1718 - val_accuracy: 0.7155 - val_loss: 1.0407 - learning_rate: 2.0000e-04
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4648 - loss: 1.4444
Epoch 2: val_accuracy improved from 0.71547 to 0.78177, saving model to cnn_densenet121_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.4804 - loss: 1.4000 - val_accuracy: 0.7818 - val_loss: 0.7209 - learning_rate: 2.0000e-04
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5780 - loss: 1.1803
Epoch 3: val_accuracy improved from 0.78177 to 0.82320, saving model to cnn_densenet121_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.5925 - loss: 1.1175 - val_accuracy: 0.8232 - val_loss: 0.5717 - learning_rate: 2.0000e-04
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━

In [17]:
test_loss, test_accuracy = hybrid_model.evaluate(test_ds)

print("Test Loss     :", test_loss)
print("Test Accuracy :", test_accuracy)

12/12 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.9123 - loss: 0.2645
Test Loss     : 0.26454389095306396
Test Accuracy : 0.9123287796974182


In [18]:
class_names = [
    "bacterial_leaf_blight",
    "brown_spot",
    "healthy",
    "leaf_blast",
    "leaf_scald",
    "leaf_smut",
    "narrow_brown_spot"
]